## Anna Giczewska ST544 - Final Project ¶
Date 5/2/2026

Short Project Description:

In [1]:
import pandas as pd
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.feature import VectorAssembler, PCA

In [2]:
spark = SparkSession.builder.appName("final_project").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 06:26:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# read the modeling data with pandas first
power_pd = pd.read_csv("power_ml_data.csv")

print(power_pd.shape)
power_pd.head()

(47174, 10)


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [4]:
# convert pandas data frame to spark data frame
power_df = spark.createDataFrame(power_pd)

power_df.printSchema()
power_df.show(5)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

In [5]:
# just to see the columns clearly
print(power_df.columns)

['Temperature', 'Humidity', 'Wind_Speed', 'General_Diffuse_Flows', 'Diffuse_Flows', 'Power_Zone_1', 'Power_Zone_2', 'Power_Zone_3', 'Month', 'Hour']


In [6]:
# cast Hour to double and create label
sql_stage = SQLTransformer(
    statement="""
    SELECT *,
           CAST(Hour AS DOUBLE) AS Hour_double,
           Power_Zone_3 AS label
    FROM __THIS__
    """
)

# binary hour variable
hour_bin = Binarizer(
    threshold=6.5,
    inputCol="Hour_double",
    outputCol="Hour_binary"
)

# one-hot encode Month
month_indexer = StringIndexer(
    inputCol="Month",
    outputCol="Month_index",
    handleInvalid="keep"
)

month_encoder = OneHotEncoder(
    inputCols=["Month_index"],
    outputCols=["Month_vec"]
)

In [7]:
# put weather variables together for PCA
weather_assembler = VectorAssembler(
    inputCols=[
        "Temperature",
        "Humidity",
        "Wind_Speed",
        "General_Diffuse_Flows",
        "Diffuse_Flows"
    ],
    outputCol="weather_features"
)

# PCA with 2 components
pca_stage = PCA(
    k=2,
    inputCol="weather_features",
    outputCol="pca_features"
)

In [8]:
# final feature vector
final_assembler = VectorAssembler(
    inputCols=[
        "pca_features",
        "Hour_binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_vec"
    ],
    outputCol="features"
)

feature_pipeline = Pipeline(stages=[
    sql_stage,
    hour_bin,
    month_indexer,
    month_encoder,
    weather_assembler,
    pca_stage,
    final_assembler
])

feature_model = feature_pipeline.fit(power_df)
feature_df = feature_model.transform(power_df)

feature_df.select("label", "features").show(5, truncate=False)

+-----------+-------------------------------------------------------------------------------------+
|label      |features                                                                             |
+-----------+-------------------------------------------------------------------------------------+
|20240.96386|(17,[0,1,3,4,8],[1.7944048636569547,-0.7412746447404452,34055.6962,16128.87538,1.0]) |
|20131.08434|(17,[0,1,3,4,8],[1.8060408300982318,-0.7108534239558476,29814.68354,19375.07599,1.0])|
|19668.43373|(17,[0,1,3,4,8],[1.8102297630563908,-0.7283113191158933,29128.10127,19006.68693,1.0])|
|18899.27711|(17,[0,1,3,4,8],[1.7986676517408848,-0.7220913032199939,28228.86076,18361.09422,1.0])|
|18442.40964|(17,[0,1,3,4,8],[1.8632872016379716,-0.7323046647776553,27335.6962,17872.34043,1.0]) |
+-----------+-------------------------------------------------------------------------------------+
only showing top 5 rows
